# Robotwin VV density plotting pipeline

Read every `step_*.csv` from `data/26Aug9-Robotwin-VV-density/attn-density-vv-10-3/` and export the final Lingbot-VA Video-Video density plot to the matching directory under `figures/`.

## Imports and project root

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if all((candidate / name).is_dir() for name in ("notebooks", "data", "figures")):
            return candidate
    raise FileNotFoundError("Could not find the jupyter-plot project root")


PROJECT_ROOT = find_project_root()
print(f"project: {PROJECT_ROOT}")

## Data and output configuration

In [ ]:
WORKSET_NAME = "26Aug9-Robotwin-VV-density"
DATASET_NAME = "attn-density-vv-10-3"
INPUT_DIR = PROJECT_ROOT / "data" / WORKSET_NAME / DATASET_NAME
OUTPUT_DIR = PROJECT_ROOT / "figures" / WORKSET_NAME
OUTPUT_STEM = OUTPUT_DIR / "lingbot_va_video_video_density"
INPUT_PATTERN = "step_*.csv"
OUTPUT_FORMATS = ("png", "pdf")
DPI = 300

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"input : {INPUT_DIR}")
print(f"output: {OUTPUT_DIR}")

## Plot configuration

In [ ]:
TITLE = "Lingbot-VA (Video-Video)"
X_LABEL = "Layer"
Y_LABEL = "Density"
Y_LIMITS = (0.0, 1.0)
FIGSIZE = (4.4, 3.35)
LINE_WIDTH = 2.2
X_GRID_STEP = 5
SHOW_LEGEND = False
COLORS = ("#E8898F", "#64A9D3", "#F2C75C", "#8BC5A1")

## Load and validate the density tables

In [ ]:
CSV_PATHS = sorted(INPUT_DIR.glob(INPUT_PATTERN))
if not CSV_PATHS:
    raise FileNotFoundError(f"No files matching {INPUT_PATTERN!r} under {INPUT_DIR}")

tables = []
for csv_path in CSV_PATHS:
    table = pd.read_csv(csv_path)
    missing_columns = {"layer", "density"} - set(table.columns)
    if missing_columns:
        raise ValueError(f"{csv_path.name}: missing columns {sorted(missing_columns)}")
    table = table[["layer", "density"]].copy()
    table["layer"] = pd.to_numeric(table["layer"], errors="raise")
    table["density"] = pd.to_numeric(table["density"], errors="raise")
    if table["layer"].duplicated().any():
        raise ValueError(f"{csv_path.name}: duplicate layer indices")
    if not np.isfinite(table[["layer", "density"]].to_numpy()).all():
        raise ValueError(f"{csv_path.name}: non-finite layer or density value")
    if not table["density"].between(*Y_LIMITS).all():
        raise ValueError(f"{csv_path.name}: density must lie within {Y_LIMITS}")
    table["step"] = csv_path.stem
    tables.append(table.sort_values("layer"))

DENSITY_TABLE = pd.concat(tables, ignore_index=True)
print(f"loaded {len(CSV_PATHS)} CSV files and {len(DENSITY_TABLE)} rows")
display(DENSITY_TABLE)

## Draw and export the figure

In [ ]:
mpl.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"],
        "font.size": 11,
        "axes.titlesize": 16,
        "axes.labelsize": 12,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "axes.linewidth": 1.1,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

fig, ax = plt.subplots(figsize=FIGSIZE)
for index, (step, table) in enumerate(DENSITY_TABLE.groupby("step", sort=True)):
    ax.plot(
        table["layer"],
        table["density"],
        color=COLORS[index % len(COLORS)],
        linewidth=LINE_WIDTH,
        alpha=0.95,
        label=step.replace("_", " "),
    )

layer_min = int(DENSITY_TABLE["layer"].min())
layer_max = int(DENSITY_TABLE["layer"].max())
layer_ticks = list(range(layer_min, layer_max + 1, 10))
if layer_max not in layer_ticks:
    layer_ticks.append(layer_max)
ax.set_xlim(layer_min - 0.7, layer_max + 0.7)
ax.set_xticks(layer_ticks)
ax.set_ylim(*Y_LIMITS)
ax.set_yticks(np.linspace(Y_LIMITS[0], Y_LIMITS[1], 6))
ax.set_title(TITLE, pad=8)
ax.set_xlabel(X_LABEL, labelpad=5)
ax.set_ylabel(Y_LABEL, labelpad=5)
ax.grid(False)
ax.yaxis.grid(True, color="#CFCFCF", linewidth=0.8, alpha=0.58)
first_grid_layer = ((layer_min // X_GRID_STEP) + 1) * X_GRID_STEP
for grid_layer in range(first_grid_layer, layer_max, X_GRID_STEP):
    ax.axvline(grid_layer, color="#CFCFCF", linewidth=0.8, alpha=0.58, zorder=1.5)
ax.set_axisbelow(True)
ax.tick_params(axis="x", direction="out", width=0, length=0, pad=7)
ax.tick_params(axis="y", direction="out", width=1.0, length=4)
for spine in ax.spines.values():
    spine.set_color("#222222")
    spine.set_linewidth(1.1)
if SHOW_LEGEND:
    ax.legend(frameon=False, fontsize=8, loc="best")

fig.tight_layout(pad=0.45)
SAVED_PATHS = []
for extension in OUTPUT_FORMATS:
    output_path = OUTPUT_STEM.with_suffix(f".{extension}")
    fig.savefig(output_path, dpi=DPI, bbox_inches="tight", pad_inches=0.03)
    SAVED_PATHS.append(output_path)
plt.show()
print(*SAVED_PATHS, sep="\n")